# RAG Pipeline Evaluation
End-to-end: Qdrant RAG + vLLM → latency + RAGAS (Vertex AI Gemini)

In [ ]:
!pip install -q vllm qdrant-client sentence-transformers openai \
            ragas langchain-google-vertexai langchain \
            matplotlib seaborn pandas

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

In [ ]:
import vertexai

# Vertex AI
GCP_PROJECT     = "your-gcp-project-id"   # <-- đổi
VERTEX_LOCATION = "us-central1"
GEMINI_MODEL    = "gemini-2.0-flash-001"

# vLLM
VLLM_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
VLLM_PORT  = 8000

# Qdrant
SNAPSHOT_PATH = "/content/drive/MyDrive/Nutrition data/nutrition_articles-2744933042503761-2026-03-30-08-11-07.snapshot"
COLLECTION    = "nutrition_articles"
TOP_K         = 5

# Embedding
EMBED_MODEL  = "intfloat/multilingual-e5-large-instruct"
QUERY_PREFIX = "Instruct: Tìm thông tin dinh dưỡng liên quan\nQuery: "

# Eval files
DRIVE    = "/content/drive/MyDrive/Nutrition data"
EVAL_FILES = [f"{DRIVE}/eval_split_{n}.jsonl" for n in range(1, 6)]

# Output
RESULTS_FILE  = f"{DRIVE}/pipeline_results.jsonl"
RAGAS_SUMMARY = f"{DRIVE}/ragas_summary.json"
PLOTS_DIR     = f"{DRIVE}/eval_plots"

RESUME = True

import os; os.makedirs(PLOTS_DIR, exist_ok=True)
vertexai.init(project=GCP_PROJECT, location=VERTEX_LOCATION)
print("Config OK")

## 1. Khởi động Qdrant

In [ ]:
import subprocess, time, requests, os

os.system("pkill -f qdrant 2>/dev/null"); time.sleep(1)

os.system("curl -L https://github.com/qdrant/qdrant/releases/latest/download/"
          "qdrant-x86_64-unknown-linux-musl.tar.gz -o /content/qdrant.tar.gz")
os.system("tar -xzf /content/qdrant.tar.gz -C /content/ qdrant")
os.system("chmod +x /content/qdrant")

qdrant_proc = subprocess.Popen(
    ["/content/qdrant"],
    stdout=open("qdrant.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

r = requests.get("http://localhost:6333/healthz")
print(f"Qdrant: {r.status_code} (PID={qdrant_proc.pid})")

In [ ]:
import requests

requests.delete(f"http://localhost:6333/collections/{COLLECTION}")

print(f"Uploading snapshot...")
with open(SNAPSHOT_PATH, "rb") as f:
    r = requests.post(
        f"http://localhost:6333/collections/{COLLECTION}/snapshots/upload?priority=snapshot",
        files={"snapshot": f},
        timeout=600,
    )
assert r.status_code == 200, f"Upload failed: {r.text}"

# Đợi index xong
import time
for _ in range(60):
    info = requests.get(f"http://localhost:6333/collections/{COLLECTION}").json()
    status = info["result"]["status"]
    count  = info["result"]["points_count"]
    if status == "green":
        print(f"Collection '{COLLECTION}': {count:,} vectors — OK")
        break
    time.sleep(5)

## 2. Load embedding model

In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

qdrant = QdrantClient(url="http://localhost:6333", timeout=60)
embed_model = SentenceTransformer(EMBED_MODEL)
print(f"Embedding model loaded: {EMBED_MODEL}")

## 3. Khởi động vLLM

In [ ]:
import subprocess, sys, time, requests, os

os.system("pkill -f vllm.entrypoints 2>/dev/null"); time.sleep(2)

vllm_proc = subprocess.Popen([
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model",                  VLLM_MODEL,
    "--port",                   str(VLLM_PORT),
    "--max-model-len",          "8192",
    "--gpu-memory-utilization", "0.85",
], stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)

print("Chờ vLLM (~3-5 phút)...")
for i in range(120):
    try:
        if requests.get(f"http://localhost:{VLLM_PORT}/health", timeout=2).status_code == 200:
            print(f"vLLM ready! ({i*5}s, PID={vllm_proc.pid})")
            break
    except: pass
    time.sleep(5)
else:
    print("TIMEOUT"); os.system("tail -20 /content/vllm.log")

## 4. Chạy pipeline: RAG + LLM + đo latency

In [ ]:
import json, time
from pathlib import Path
from openai import OpenAI

SYSTEM_PROMPT = (
    "Bạn là chuyên gia tư vấn dinh dưỡng qua giọng nói. Tuân thủ:\n"
    "1. Dựa vào tài liệu được cung cấp để trả lời.\n"
    "2. Bắt đầu bằng 'Chào bạn,', dùng câu ngắn, dễ nghe.\n"
    "3. Nếu không có thông tin: 'Tôi không có thông tin về vấn đề này'.\n"
    "4. Kết thúc: 'Để được tư vấn chính xác, bạn nên gặp bác sĩ dinh dưỡng.'\n"
    "/no_think"
)

client = OpenAI(base_url=f"http://localhost:{VLLM_PORT}/v1", api_key="EMPTY")

# Load samples
all_samples = []
for path in EVAL_FILES:
    if Path(path).exists():
        all_samples += [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
print(f"Total samples: {len(all_samples)}")

# Resume
done_ids = set()
if RESUME and Path(RESULTS_FILE).exists():
    with open(RESULTS_FILE, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                done_ids.add(json.loads(line)["id"])
    print(f"Resume: {len(done_ids)} câu đã có, bỏ qua.")

pending = [s for s in all_samples if s["id"] not in done_ids]
print(f"Cần chạy: {len(pending)} câu\n")

with open(RESULTS_FILE, "a", encoding="utf-8") as fout:
    for i, s in enumerate(pending):
        print(f"  [{i+1}/{len(pending)}] {s['id']} ...", end="", flush=True)
        try:
            t_total = time.perf_counter()

            # ── RAG: embed + search Qdrant ──
            t_rag = time.perf_counter()
            q_vec = embed_model.encode(
                [QUERY_PREFIX + s["question"]], normalize_embeddings=True
            )[0].tolist()
            hits = qdrant.query_points(
                collection_name=COLLECTION, query=q_vec,
                limit=TOP_K, with_payload=True,
            )
            rag_ms = (time.perf_counter() - t_rag) * 1000
            contexts = [p.payload.get("text", "") for p in hits.points]

            # ── LLM: generate (streaming để đo TTFT) ──
            context_str = "\n\n".join(contexts)
            t_llm = time.perf_counter()
            stream = client.chat.completions.create(
                model=VLLM_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content":
                        f"Tài liệu:\n{context_str}\n---\nCâu hỏi: {s['question']}"},
                ],
                temperature=0.0,
                max_tokens=1024,
                stream=True,
            )
            ttft_ms, parts = None, []
            for chunk in stream:
                text = chunk.choices[0].delta.content or ""
                if text and ttft_ms is None:
                    ttft_ms = (time.perf_counter() - t_llm) * 1000
                parts.append(text)

            answer   = "".join(parts).strip()
            llm_ms   = (time.perf_counter() - t_llm) * 1000
            total_ms = (time.perf_counter() - t_total) * 1000

            record = {
                "id": s["id"], "split": s.get("split"), "source": s.get("source"),
                "question": s["question"], "answer": answer,
                "contexts": contexts,      "reference": s["answer"],
                "latency": {
                    "rag_ms"  : round(rag_ms, 1),
                    "ttft_ms" : round(ttft_ms or 0, 1),
                    "llm_ms"  : round(llm_ms, 1),
                    "total_ms": round(total_ms, 1),
                },
            }
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            fout.flush()
            done_ids.add(s["id"])
            print(f" rag={rag_ms:.0f}ms  ttft={ttft_ms:.0f}ms  total={total_ms:.0f}ms")

        except Exception as e:
            print(f" ERROR: {e}")

print(f"\nDone → {RESULTS_FILE}")

In [ ]:
import os
os.system("pkill -f vllm.entrypoints")
print("vLLM stopped.")

## 5. RAGAS Evaluation (Vertex AI Gemini)

In [ ]:
import json
from ragas import evaluate
from ragas.metrics import AnswerRelevancy, Faithfulness, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings

results = [json.loads(l) for l in open(RESULTS_FILE, encoding="utf-8") if l.strip()]
print(f"Loaded {len(results)} results")

dataset = EvaluationDataset([
    SingleTurnSample(
        user_input=r["question"],
        response=r["answer"],
        retrieved_contexts=r["contexts"],
        reference=r["reference"],
    )
    for r in results
    if r.get("answer") and r.get("contexts") and r.get("reference")
])
print(f"RAGAS dataset: {len(dataset)} samples\n")

judge_llm = LangchainLLMWrapper(ChatVertexAI(model_name=GEMINI_MODEL, temperature=0))
judge_emb = LangchainEmbeddingsWrapper(VertexAIEmbeddings(model_name="text-embedding-004"))

metrics = [
    AnswerRelevancy(llm=judge_llm, embeddings=judge_emb),
    Faithfulness(llm=judge_llm),
    ContextPrecision(llm=judge_llm),
    ContextRecall(llm=judge_llm),
]

print("Đang chạy RAGAS...")
eval_df = evaluate(dataset=dataset, metrics=metrics).to_pandas()

metric_cols = [c for c in eval_df.columns
               if c not in ("user_input","response","retrieved_contexts","reference")]
summary = {c: round(float(eval_df[c].mean()), 4)
           for c in metric_cols if not eval_df[c].isna().all()}

print("\n=== RAGAS Scores ===")
for m, s in summary.items():
    print(f"  {m:<25} {s:.4f}  " + "█" * int(s * 20))

# Gắn RAGAS scores vào results để vẽ biểu đồ
for i, r in enumerate(results):
    if i < len(eval_df):
        r["ragas"] = {c: (round(float(eval_df.iloc[i][c]), 4)
                          if eval_df.iloc[i][c] == eval_df.iloc[i][c] else None)
                      for c in metric_cols}

with open(RAGAS_SUMMARY, "w", encoding="utf-8") as f:
    json.dump({"n_samples": len(dataset), "gen_model": VLLM_MODEL,
               "judge_model": GEMINI_MODEL, "ragas": summary},
              f, ensure_ascii=False, indent=2)
print(f"\nSummary → {RAGAS_SUMMARY}")

## 6. Biểu đồ

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_theme(style="whitegrid", palette="muted")

df = pd.DataFrame([
    {**r["latency"], "source": r.get("source", "?"), "split": r.get("split", "?")}
    for r in results if r.get("latency")
])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Latency Distribution", fontsize=14, fontweight="bold")

for ax, col, label, color in zip(
    axes,
    ["rag_ms", "ttft_ms", "total_ms"],
    ["RAG (ms)", "TTFT (ms)", "Total (ms)"],
    ["#4C72B0", "#DD8452", "#55A868"],
):
    sns.boxplot(y=df[col], ax=ax, color=color, width=0.4)
    ax.set_title(label)
    ax.set_ylabel("ms")
    p50 = df[col].median()
    p95 = df[col].quantile(0.95)
    ax.axhline(p50, color="red",   linestyle="--", linewidth=1, label=f"p50={p50:.0f}")
    ax.axhline(p95, color="orange",linestyle="--", linewidth=1, label=f"p95={p95:.0f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/latency_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Latency stats table
print("\n=== Latency Stats (ms) ===")
print(f"{'Metric':<12} {'mean':>8} {'p50':>8} {'p95':>8} {'min':>8} {'max':>8}")
print("-" * 48)
for col, label in [("rag_ms","RAG"), ("ttft_ms","TTFT"), ("total_ms","Total")]:
    v = df[col]
    print(f"{label:<12} {v.mean():>8.0f} {v.median():>8.0f} "
          f"{v.quantile(0.95):>8.0f} {v.min():>8.0f} {v.max():>8.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("RAGAS Evaluation Scores", fontsize=14, fontweight="bold")

# Overall scores bar chart
ax = axes[0]
metrics_names = list(summary.keys())
scores = [summary[m] for m in metrics_names]
colors = ["#4C72B0","#DD8452","#55A868","#C44E52"]
bars = ax.barh(metrics_names, scores, color=colors[:len(metrics_names)], height=0.5)
ax.set_xlim(0, 1)
ax.set_xlabel("Score")
ax.set_title("Overall RAGAS Scores")
for bar, score in zip(bars, scores):
    ax.text(score + 0.01, bar.get_y() + bar.get_height()/2,
            f"{score:.4f}", va="center", fontsize=10)

# Per-source breakdown
ax2 = axes[1]
ragas_df = pd.DataFrame([
    {"source": r.get("source","?"), **r["ragas"]}
    for r in results if r.get("ragas")
])
if not ragas_df.empty and "source" in ragas_df.columns:
    per_source = ragas_df.groupby("source")[metric_cols].mean().round(4)
    per_source.plot(kind="bar", ax=ax2, colormap="Set2", width=0.6)
    ax2.set_title("RAGAS Scores by Source")
    ax2.set_xlabel("Source")
    ax2.set_ylabel("Score")
    ax2.set_ylim(0, 1)
    ax2.legend(loc="lower right", fontsize=8)
    ax2.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/ragas_scores.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
ragas_df2 = pd.DataFrame([
    {"split": str(r.get("split","?")), **r["ragas"]}
    for r in results if r.get("ragas")
])

if not ragas_df2.empty:
    pivot = ragas_df2.groupby("split")[metric_cols].mean().round(4)

    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGn",
                vmin=0, vmax=1, linewidths=0.5, ax=ax)
    ax.set_title("RAGAS Scores per Split", fontsize=13, fontweight="bold")
    ax.set_xlabel("Metric")
    ax.set_ylabel("Split")
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/ragas_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()

print(f"\nPlots saved → {PLOTS_DIR}")